# 3D DPC Illumination Design

This notebook demonstrates physics-based learning to identify optimal LED array illumination patterns for 3D Differential Phase Contrast (3D DPC) microscopy.

Three experiments are covered:
1. Random initialization, noiseless optimization
2. Random initialization, with noise (same total exposure)
3. Regularization: multiple z-sampling, pseudo-binary patterns

Results are saved to `./design_results/` and visualized inline.

**Setup:** `uv venv && uv sync && source .venv/bin/activate && uv run jupyter notebook 3D_DPC_illumination_design.ipynb`

In [ ]:
# Google colab only
import os
os.system('git clone https://github.com/rmcao/self-optimized-dpc.git')
os.chdir('/content/self-optimized-dpc')

In [ ]:
%matplotlib inline
import numpy as np
import json, os
from design_3ddpc_illumination import DesignMotion3DDPCIllumination
from solver_3ddpc import Solver3DDPC
from visualization import visualize_patterns, visualize_transfer_functions

## General Setup

In [ ]:
# Optical parameters
wavelength     = 0.532    # µm
mag            = 40
na             = 0.65
pixel_size_cam = 6.5      # µm (camera pixel size)
pixel_size     = pixel_size_cam / mag  # effective pixel size in µm
RI_medium      = 1.58     # background refractive index
pixel_size_z   = 0.8      # axial pixel size in µm

In [ ]:
# Load LED positions from JSON
led_pos_json_file = 'led_array_pos_na_z65mm.json'
with open(led_pos_json_file) as f:
    d_led_pos = json.load(f)
source_list_na_design = d_led_pos['led_position_list_na']
source_list_fxfy = np.array([source_list_na_design[k] for k in source_list_na_design.keys()]) / wavelength
print(f'Loaded {len(source_list_na_design)} LED positions')

In [ ]:
# Create design object and generate LED illumination source mask
design_obj = DesignMotion3DDPCIllumination(
    (200, 200, 60), wavelength, na, pixel_size, pixel_size_z, RI_medium)
source = design_obj.get_illu_pattern_bases_LEDdome(source_list_fxfy)
print(f'Source mask shape: {source.shape}, active LEDs: {int(source.sum())}')

In [ ]:
# Build high-resolution visualization helper (650×650 pixels at 2× pixel size)
solver_3ddpc_highres = Solver3DDPC(
    np.ones((1, 10, 650, 650)).transpose(2, 3, 1, 0),
    wavelength, na, 0,
    pixel_size=pixel_size * 2,
    pixel_size_z=pixel_size_z,
    rotation=[0],
    RI_medium=RI_medium,
    dim_z=4)

# Collect all LEDs within NA
num_led = 581
list_led_num_in_na = []
for i in range(num_led):
    if (source_list_na_design[str(i)][0]**2 + source_list_na_design[str(i)][1]**2) <= na**2:
        list_led_num_in_na.append(i)
all_source = solver_3ddpc_highres.get_illu_pattern_bases_LEDdome(
    np.array([source_list_na_design[str(k)] for k in list_led_num_in_na]) / wavelength,
    large_led=True)

def source2source_highres(s):
    """Map a 200×200 source pattern to 650×650 high-res for visualization."""
    s_highres = np.zeros_like(all_source)
    for i in range(num_led):
        ind_source = design_obj.get_illu_pattern_bases_LEDdome(
            [np.array(source_list_na_design[str(i)]) / wavelength])
        ind_source_highres = solver_3ddpc_highres.get_illu_pattern_bases_LEDdome(
            [np.array(source_list_na_design[str(i)]) / wavelength], large_led=True)
        s_highres += np.sum(ind_source * s) * ind_source_highres
    return s_highres * 3 + all_source

print('High-res visualization helper ready.')

In [ ]:
# Generate synthetic test objects and random initial illumination coefficients
num_pat = [1, 2, 3, 4, 5, 6, 7, 8]

design_obj_numpat = DesignMotion3DDPCIllumination(
    (200, 200, 60), wavelength, na, pixel_size, pixel_size_z,
    RI_medium, source_mask=source, num_illu=4)
v_objects, _ = design_obj_numpat.generate_object_scattering_potential(
    16, delta_RI=7.5e-3, delta_attenu=1e-3, add_noise=True)

pixel_rand_init_coef = (np.random.uniform(0, 1.0, size=(max(num_pat), 200, 200))
                        * design_obj_numpat.source_mask[np.newaxis, :, :])

os.makedirs('./design_results', exist_ok=True)
print(f'Generated {len(v_objects)} synthetic objects, init coef shape: {pixel_rand_init_coef.shape}')

## 1. Random Init, Noiseless

Optimize illumination patterns without photon shot noise (`intensity_coef=0.0`). This provides an upper bound on achievable reconstruction quality.

In [ ]:
opt_pat_noiseless, error_noiseless = [], []
for num_illu in num_pat:
    design_obj_numpat = DesignMotion3DDPCIllumination(
        (200, 200, 60), wavelength, na, pixel_size, pixel_size_z,
        RI_medium, source_mask=source, num_illu=num_illu, intensity_coef=0.0)
    opt, err = design_obj_numpat.optimize_illu_pattern_tikhonov(
        False,
        init_coef=pixel_rand_init_coef[:num_illu, :, :],
        iters=200, lr=2e-2,
        V_objects=v_objects,
        imperfect_mode=False,
        dims_z=[60])
    opt_pat_noiseless.append(opt)
    error_noiseless.append(err)
    print(f'num_illu={num_illu} done, final loss={err[-1]:.4f}')

In [ ]:
# Save results
np.savez('./design_results/noiseless_optimization.npz',
         opt_pat_noiseless=np.array(opt_pat_noiseless, dtype=object),
         error_noiseless=np.array(error_noiseless, dtype=object),
         pixel_rand_init_coef=pixel_rand_init_coef,
         v_objects=np.array(v_objects))
print('Saved: ./design_results/noiseless_optimization.npz')

In [ ]:
# Visualize optimized patterns for 4-pattern case (index 3)
visualize_patterns(opt_pat_noiseless[3], source2source_highres,
                   title='Noiseless – 4 patterns')

## 2. Random Init, With Noise (Same Total Exposure)

`intensity_coef = 2.0 / num_illu` keeps total illumination constant across different numbers of patterns.

In [ ]:
opt_pat_noisy, error_noisy = [], []
for num_illu in num_pat:
    intensity_coef = 2.0 / num_illu
    design_obj_numpat = DesignMotion3DDPCIllumination(
        (200, 200, 60), wavelength, na, pixel_size, pixel_size_z,
        RI_medium, source_mask=source, num_illu=num_illu,
        intensity_coef=intensity_coef)
    coef, err = design_obj_numpat.optimize_illu_pattern_tikhonov(
        False,
        init_coef=pixel_rand_init_coef[:num_illu, :, :],
        iters=200, lr=2e-2,
        V_objects=v_objects,
        imperfect_mode=False,
        dims_z=[60])
    opt_pat_noisy.append(coef)
    error_noisy.append(err)
    print(f'num_illu={num_illu}, intensity_coef={intensity_coef:.3f}, final loss={err[-1]:.4f}')

In [ ]:
# Save results
np.savez('./design_results/noisy_optimization.npz',
         opt_pat_noisy=np.array(opt_pat_noisy, dtype=object),
         error_noisy=np.array(error_noisy, dtype=object))
print('Saved: ./design_results/noisy_optimization.npz')

In [ ]:
# Visualize optimized patterns for 4-pattern case (index 3)
visualize_patterns(opt_pat_noisy[3], source2source_highres,
                   title='With Noise – 4 patterns')

## 3. Regularization: Multiple Sampling, Pseudo-Binary Patterns

Uses `imperfect_mode=True` which applies:
- Multiple z-sampling depths (`dims_z=[60, 55, 50]`)
- Random spatial shifts to simulate LED positioning errors
- Occasional binarization during training to encourage binary-like patterns

`intensity_coef=1.0` with `num_illu=4`.

In [ ]:
num_illu = 4
design_obj_reg = DesignMotion3DDPCIllumination(
    (200, 200, 60), wavelength, na, pixel_size, pixel_size_z,
    RI_medium, source_mask=source, num_illu=num_illu, intensity_coef=1.0)

opt_pat_reg, error_reg = design_obj_reg.optimize_illu_pattern_tikhonov(
    False,
    init_coef=pixel_rand_init_coef[:num_illu, :, :],
    iters=300, lr=2e-2,
    V_objects=v_objects,
    imperfect_mode=True,
    dims_z=[60, 55, 50],
    shift_range=3)
print(f'Regularized optimization done, final loss={error_reg[-1]:.4f}')

In [ ]:
# Save results
np.savez('./design_results/regularized_optimization.npz',
         opt_pat_reg=np.array(opt_pat_reg),
         error_reg=np.array(error_reg))
print('Saved: ./design_results/regularized_optimization.npz')

In [ ]:
# Visualize final optimized patterns
visualize_patterns(opt_pat_reg, source2source_highres,
                   title='Regularized – 4 patterns (intensity_coef=1.0)')

## Phase Transfer Function Visualization

Compute the 3D weak object transfer functions (WOTFs) for the regularized optimized patterns and display phase TF slices at increasing axial frequencies. The missing cone region (where 3D DPC has no sensitivity) is overlaid as a white circle.

In [ ]:
# Compute WOTFs for the binarized regularized patterns
H_real, H_imag = design_obj_reg.WOTFGen(np.array(opt_pat_reg[-1]) > 0.7)
print(f'H_real shape: {H_real.shape}  (num_illu, H, W, Z)')

In [ ]:
# Visualize phase transfer functions with missing cone overlay
visualize_transfer_functions(
    H_real, opt_pat_reg, design_obj_reg,
    source2source_highres, na, wavelength,
    fz_ind_min=18, fz_step=6, fz_num=5,
    clim=(-0.1, 0.1),
    save_path='transfer_function_visualization.pdf')
print('Saved: transfer_function_visualization.pdf')